# Kaggle GPU verification - IBM AML GNN

Targeted verification для курсовой: проверить ключевые standalone GNN на Kaggle GPU и пересобрать итоговые таблицы/графики. В текущем Kaggle PyTorch P100 может быть несовместима (sm_60), поэтому предпочтительно выбрать T4 x2 или L4.

Что запускается по умолчанию:
- `ibm_gine_fulldata`
- `ibm_multignn_fulldata`
- `ibm_pna_fulldata`
- `src.compare --ibm` для сборки Markdown/PNG

Опционально можно включить multi-seed и гибрид `GNN embedding -> XGBoost`. Для полного гибрида на 5.08M ребер нужно много времени и RAM; включайте его отдельным запуском.

## 0. Kaggle setup

Перед запуском:
1. Notebook settings -> Accelerator -> `GPU T4 x2` (рекомендуется) или `GPU L4`. P100 использовать только если установлен PyTorch с поддержкой sm_60.
2. Notebook settings -> Internet -> `On`, если репозиторий или PyG wheels не прикреплены как dataset.
3. Add Data -> IBM AML dataset с файлами `HI-Small_Trans.csv` и `HI-Small_Patterns.txt`.
4. Если репозиторий приватный и git clone не доступен, прикрепите zip/source как Kaggle Dataset и укажите `REPO_INPUT_DIR` ниже.

In [ ]:
import os, sys, subprocess, shutil, json, glob, textwrap, pathlib
from pathlib import Path

import torch
assert torch.cuda.is_available(), "GPU не виден. Включи Accelerator -> GPU P100."
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
name = torch.cuda.get_device_name(0)
if "P100" in name:
    print("WARNING: P100 (sm_60) может быть несовместима с текущим Kaggle PyTorch. Если видишь CUDA capability warning, переключись на T4 x2 или L4.")
if torch.cuda.device_count() > 1:
    print("GPUs visible:", torch.cuda.device_count(), "(код использует одну GPU; T4 x2 полезна главным образом совместимостью/доступностью, не удвоением памяти)")

## 1. Репозиторий

По умолчанию notebook клонирует GitHub repo. Если GitHub недоступен, загрузите исходники как Kaggle Dataset и задайте `REPO_INPUT_DIR`.

In [ ]:
REPO_URL = "https://github.com/Ezzenin/gnn-aml-transaction-graphs.git"
REPO = Path("/kaggle/working/gnn-aml-transaction-graphs")

# Optional source path.
# REPO_INPUT_DIR = "/kaggle/input/gnn-aml-transaction-graphs"
REPO_INPUT_DIR = None

if REPO.exists():
    print("repo exists:", REPO)
elif REPO_INPUT_DIR:
    shutil.copytree(REPO_INPUT_DIR, REPO)
    print("copied repo from dataset:", REPO_INPUT_DIR)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("cwd:", Path.cwd())
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip() if (REPO/'.git').exists() else "no git metadata")

## 2. Зависимости

Kaggle уже содержит PyTorch CUDA. PyG sampler packages (`pyg-lib`, `torch-scatter`, `torch-sparse`) ставятся под текущие версии torch/cuda.

In [ ]:
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print("PyG wheel index:", wheel_url)

base_pkgs = [
    "torch_geometric", "xgboost", "scikit-learn", "pandas", "numpy",
    "networkx", "matplotlib", "pyyaml", "tqdm"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *base_pkgs], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyg-lib", "torch-scatter", "torch-sparse", "-f", wheel_url
], check=True)

import torch_geometric
print("torch_geometric:", torch_geometric.__version__)
try:
    import pyg_lib, torch_sparse
    print("sampler deps OK")
except Exception as e:
    print("WARNING: sampler deps import failed:", repr(e))

## 3. Данные IBM AML

Notebook ищет `HI-Small_Trans.csv` и `HI-Small_Patterns.txt` рекурсивно в `/kaggle/input` и делает symlink/copy в `data/ibm_aml/`.

In [ ]:
from pathlib import Path
import os, shutil, glob

def find_one(pattern):
    matches = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Не найден {pattern} в /kaggle/input. Add Data с IBM AML dataset.")
    matches = sorted(matches, key=len)
    print(pattern, "->", matches[0])
    return Path(matches[0])

trans = find_one("HI-Small_Trans.csv")
patterns = find_one("HI-Small_Patterns.txt")

dst_dir = Path("data/ibm_aml")
dst_dir.mkdir(parents=True, exist_ok=True)
for src, name in [(trans, "HI-Small_Trans.csv"), (patterns, "HI-Small_Patterns.txt")]:
    dst = dst_dir / name
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)
    print(dst, "size GB", dst.stat().st_size / 1e9)

!ls -lh data/ibm_aml

## 4. План проверки

Для полной multi-seed проверки поменяйте `SEEDS = [42, 43, 44]`. На P100 это может занять много часов. По умолчанию запускается только seed 42.

`GPU_SAFE_BATCH=True` снижает batch для PNA до 2048. Это не меняет архитектуру, но может немного изменить обучение из-за другого batching/sampling.

In [ ]:
SEEDS = [42]  # seeds: [42, 43, 44]
RUN_GINE = True
RUN_MULTIGNN = True
RUN_PNA = True

# Full hybrid extraction is heavy.
RUN_HYBRID_FULL = False

# Smoke mode. Set None for final run.
MAX_ROWS = None  # 500_000 for smoke

GPU_SAFE_BATCH = True
PNA_BATCH_SIZE = 2048

selected = []
if RUN_GINE: selected.append("ibm_gine_fulldata")
if RUN_MULTIGNN: selected.append("ibm_multignn_fulldata")
if RUN_PNA: selected.append("ibm_pna_fulldata")
print("configs:", selected)
print("seeds:", SEEDS)

## 5. Генерация временных конфигов

Seed 42 пишет стандартные `output_name`, чтобы `src.compare --ibm` подхватил результаты. Остальные seed пишутся как `<name>_seed<seed>`.

In [ ]:
import yaml, copy
from pathlib import Path

TMP_CFG_DIR = Path("configs/kaggle_tmp")
TMP_CFG_DIR.mkdir(parents=True, exist_ok=True)

def make_config(base_name, seed):
    src = Path("configs") / f"{base_name}.yaml"
    with open(src, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["seed"] = int(seed)
    cfg["device"] = "cuda"
    cfg.setdefault("dataset", {})["root"] = "data/ibm_aml"
    if MAX_ROWS is not None:
        cfg["dataset"]["max_rows"] = int(MAX_ROWS)
    cfg["output_name"] = base_name if seed == 42 else f"{base_name}_seed{seed}"
    cfg["checkpoint_dir"] = "checkpoints/kaggle"
    cfg["save_checkpoint"] = True
    if GPU_SAFE_BATCH and base_name == "ibm_pna_fulldata":
        cfg.setdefault("train", {})["batch_size"] = int(PNA_BATCH_SIZE)
    out = TMP_CFG_DIR / f"{base_name}_seed{seed}.yaml"
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    return out

tmp_configs = []
for seed in SEEDS:
    for base in selected:
        tmp_configs.append(make_config(base, seed))

print("generated:")
for p in tmp_configs:
    print(" -", p)

## 6. Запуск GNN verification

Если PNA падает с OOM, уменьшите `PNA_BATCH_SIZE` до 1024 и перезапустите генерацию конфигов.

In [ ]:
import subprocess, sys, time

for cfg in tmp_configs:
    print("\n" + "="*90)
    print("RUN", cfg)
    t0 = time.time()
    subprocess.run([sys.executable, "-m", "src.train_edge", "--config", str(cfg)], check=True)
    print("elapsed min:", round((time.time() - t0) / 60, 2))

## 7. Сводка seed-прогонов

In [ ]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob("results/ibm_*fulldata*_metrics.json")):
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    test = d.get("test_metrics", {}) or {}
    cfg = d.get("config", {}) or {}
    rows.append({
        "file": Path(path).name,
        "output_name": cfg.get("output_name"),
        "seed": cfg.get("seed"),
        "model": d.get("model_type") or (cfg.get("model", {}) or {}).get("type"),
        "auc_pr": test.get("auc_pr"),
        "f1": test.get("f1"),
        "roc_auc": test.get("roc_auc"),
        "r_at_p90": test.get("recall_at_precision_90"),
        "recall": test.get("recall"),
    })

df = pd.DataFrame(rows).sort_values(["auc_pr"], ascending=False, na_position="last")
display(df)
df.to_csv("results/kaggle_gpu_seed_summary.csv", index=False)

## 8. Опционально: полный гибрид GNN embedding -> XGBoost

Это самый тяжелый шаг. Запускайте после проверки, что нужный checkpoint существует. Для Multi-GNN seed 42 путь обычно `checkpoints/kaggle/ibm_multignn_fulldata.pt`.

In [ ]:
if RUN_HYBRID_FULL:
    ckpt = Path("checkpoints/kaggle/ibm_multignn_fulldata.pt")
    if not ckpt.exists():
        ckpt = Path("checkpoints/ibm_multignn_fulldata.pt")
    assert ckpt.exists(), f"checkpoint not found: {ckpt}"
    out_json = "results/ibm_hybrid_gnn_xgb_metrics.json"
    subprocess.run([sys.executable, "scripts/hybrid_gnn_xgb.py", "-", str(ckpt), out_json], check=True)
else:
    print("RUN_HYBRID_FULL=False; пропускаю полный гибрид.")

## 9. Пересборка итоговых таблиц и графиков

In [ ]:
subprocess.run([sys.executable, "-m", "src.compare", "--ibm"], check=True)
print("main artifacts:")
for p in [
    "results/ibm_all_variants.md",
    "results/ibm_all_variants_ranking.png",
    "results/ibm_family_best.png",
    "results/ibm_ablation_heatmap.png",
    "results/ibm_ladder.md",
    "results/per_pattern.md",
]:
    print(p, Path(p).exists())

## 10. Упаковка результатов для скачивания

In [ ]:
import zipfile, os
zip_path = Path("/kaggle/working/kaggle_gpu_results.zip")
patterns = [
    "results/ibm_*metrics.json",
    "results/ibm_*.md",
    "results/ibm_*.png",
    "results/per_pattern.*",
    "results/kaggle_gpu_seed_summary.csv",
    "checkpoints/kaggle/*.pt",
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in patterns:
        for f in glob.glob(pat):
            z.write(f)
print("saved", zip_path, "size MB", zip_path.stat().st_size / 1e6)